# 03 - Salary Analysis

Analyse salary distributions by role, location, experience, and skills.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

from src.data_loader import load_raw_data, detect_and_rename_columns
from src.data_cleaning import clean_data
from src.skill_extractor import extract_skills
from src.feature_engineering import engineer_features
from src import analytics

In [ ]:
df = load_raw_data()
df = detect_and_rename_columns(df)
cleaned = clean_data(df)
cleaned = extract_skills(cleaned)
processed = engineer_features(cleaned)

sal = processed.dropna(subset=['salary_average']).copy()
sal['salary_lpa'] = sal['salary_average'] / 100000
print(f"Salary records available: {len(sal)} / {len(processed)}")

## Salary Distribution

In [ ]:
sal['salary_lpa'].hist(bins=40, figsize=(10, 5), title='Salary distribution (LPA)')
plt.tight_layout()
plt.show()
sal['salary_lpa'].describe()

## Salary by Role

In [ ]:
by_role = analytics.get_salary_by_role(processed)
by_role.plot(kind='barh', x='job_role', y='avg_salary', figsize=(10, 6), title='Average salary by role')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Salary by Location

In [ ]:
by_loc = analytics.get_salary_by_location(processed).head(12)
by_loc.plot(kind='barh', x='location', y='avg_salary', figsize=(10, 6), title='Average salary by location')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Salary by Experience

In [ ]:
by_exp = analytics.get_salary_by_experience(processed)
by_exp.plot(kind='bar', x='experience', y='avg_salary', figsize=(10, 5), title='Average salary by experience')
plt.tight_layout()
plt.show()

## Skills vs Salary

In [ ]:
# Average salary per skill (where the skill is required)
skill_salaries = {}
for _, row in sal.iterrows():
    for skill in row['extracted_skills']:
        skill_salaries.setdefault(skill, []).append(row['salary_average'])

data = [{'skill': s, 'avg_salary': np.mean(vals), 'count': len(vals)}
        for s, vals in skill_salaries.items() if len(vals) >= 10]
skill_sal = pd.DataFrame(data).sort_values('avg_salary', ascending=False)
skill_sal.head(15).plot(kind='barh', x='skill', y='avg_salary', figsize=(10, 7), title='Skills with highest avg salary')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: experience vs salary
sc = sal.dropna(subset=['experience_min'])
sns.scatterplot(data=sc, x='experience_min', y='salary_lpa', hue='standardized_job_title', alpha=0.6)
plt.title('Experience vs Salary by role')
plt.tight_layout()
plt.show()